# Experiment 08 — Final privacy–utility analysis

**Experiment 06 accepted for final analysis, not as proof of an optimal setting.**
This notebook follows `ROADMAP_REVISED_V2`. Experiment 07's optional feature-noise comparator is skipped.

**Run in Colab:** use the same `ML-DP-NID` Drive project, CPU runtime, and choose **Runtime → Run all**.
No target models, shadows, or attackers are trained. No thresholds, splits, or privacy settings are tuned.

Inputs: the accepted `results/repeated_runs/` files (or `experiment06_evidence.zip`) and three
seed-42 target-score caches already created by Experiment 06. The ZIP alone supports the core
analysis; member/nonmember distribution figures additionally require those score caches.
If caches are unavailable, a clearly marked PARTIAL evidence ZIP is still exported with recovery paths.
Do not rerun training to recover a missing plotting input.

Outputs: paper tables, nine roadmap figures plus false-alarm diagnostics, plot data, interpretation,
and a checksummed `experiment08_evidence.zip` under `results/final_analysis/`.

**Accepted scientific boundary:** ε≈4 has no established recall advantage over non-private training;
false alarms rise and average precision falls. MIA AUCs are close to 0.5, but some *across-seed*
intervals exclude 0.5. Paired AUC comparisons do not show leakage reduction. The small positive
ε≈2 label-aware advantage difference must also be disclosed. These are unadjusted, conditional
five-seed intervals on a fixed evaluation sample, not population privacy guarantees.


## 1. Imports and accepted evidence identity


In [1]:
from pathlib import Path
import os, io, json, hashlib, zipfile, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180, "font.size": 10})
ACCEPTED_MANIFEST_SHA256 = 'f1d93ba4362ebe9308fd7e47bcf111f90a9c8d2e8ebd7e3691e5f1686a16b570'
PROTOCOL = "ROADMAP_REVISED_V2"
ANALYSIS_REVISION = "final_analysis_v1"


## 2. Locate the existing experiment files


In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

PROJECT_DIR = Path(os.environ.get("ML_DP_NID_DIR", "/content/drive/MyDrive/ML-DP-NID"))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
EXP06_DIR = PROJECT_DIR / "results" / "repeated_runs"
OUTPUT_DIR = PROJECT_DIR / "results" / "final_analysis"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
PLOT_DIR = OUTPUT_DIR / "plot_data"
for directory in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

explicit = os.environ.get("EXPERIMENT06_EVIDENCE")
if explicit:
    evidence_source = Path(explicit)
elif (EXP06_DIR / "repeated_runs_manifest.json").exists():
    evidence_source = EXP06_DIR
elif (EXP06_DIR / "experiment06_evidence.zip").exists():
    evidence_source = EXP06_DIR / "experiment06_evidence.zip"
elif (PROJECT_DIR / "experiment06_evidence.zip").exists():
    evidence_source = PROJECT_DIR / "experiment06_evidence.zip"
else:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Set EXPERIMENT06_EVIDENCE to the accepted ZIP, or restore results/repeated_runs/.")
    print("Upload the accepted experiment06_evidence.zip. Distribution figures still need the Drive score caches.")
    uploaded = files.upload()
    if "experiment06_evidence.zip" not in uploaded:
        raise FileNotFoundError("Expected experiment06_evidence.zip")
    evidence_source = Path("experiment06_evidence.zip")
print("Evidence source:", evidence_source)
print("Output directory:", OUTPUT_DIR)


Mounted at /content/drive
Evidence source: /content/drive/MyDrive/ML-DP-NID/results/repeated_runs
Output directory: /content/drive/MyDrive/ML-DP-NID/results/final_analysis


## 3. Independently validate input hashes, run coverage and statistics

The following audit is embedded so this Colab notebook does not require cloning the repository.
It is the same audit as `scripts/audit_experiment06.py`. It checks evidence consistency and
recomputes summaries from per-run records; it does not reproduce the neural predictions.


In [3]:
"""Read-only audit of Experiment 06 evidence; no neural-model training.

CLI: python scripts/audit_experiment06.py /path/to/experiment06_evidence.zip
Checks internal integrity and recalculates statistics from per-run tables.
Does not independently reproduce neural predictions or privacy accounting.
"""
import hashlib
import io
import json
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

SEEDS = [42, 52, 62, 72, 82]
CONDITIONS = ['non_private', 'dp_eps_4', 'dp_eps_2']
THREATS = ['score_only_black_box', 'label_aware_audit']
PRIMARY = dict(zip(THREATS, ['logistic_regression', 'loss_threshold']))
IDS_METRICS = ['recall', 'fnr', 'f1', 'fpr', 'precision', 'pr_auc', 'threshold']
MIA_METRICS = ['mia_auc', 'mia_advantage', 'mia_balanced_accuracy', 'tpr_at_1pct_fpr', 'tpr_at_5pct_fpr']
T_CRITICAL = 2.7764451051977987


def require(test, message):
    if not bool(test):
        raise ValueError(message)


def close(a, b, message, atol=1e-12):
    require(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), rtol=0, atol=atol, equal_nan=True), message)


def read_bundle(source):
    source = Path(source)
    if source.is_dir():
        manifest = json.loads((source / 'repeated_runs_manifest.json').read_text())
        names = list(manifest['output_sha256']) + ['repeated_runs_manifest.json']
        require(all(Path(n).name == n for n in names), 'Invalid artifact filename')
        return {n: (source / n).read_bytes() for n in names}
    with zipfile.ZipFile(source) as archive:
        names = archive.namelist()
        require(len(names) == len(set(names)), 'Duplicate ZIP entries')
        require(all(Path(n).name == n for n in names), 'ZIP must have flat artifact names')
        return {n: archive.read(n) for n in names}


def stats(values):
    values = np.asarray(values, dtype=float)
    require(len(values) == 5 and np.isfinite(values).all(), 'Expected five finite seed values')
    mean = values.mean()
    sd = values.std(ddof=1)
    half = T_CRITICAL * sd / np.sqrt(5)
    return [mean, sd, mean-half, mean+half]


def audit_bundle(blobs):
    m = json.loads(blobs['repeated_runs_manifest.json'])
    require(m['experiment'] == '06_repeated_runs_stability', 'Wrong experiment')
    require(m['protocol_version'] == 'ROADMAP_REVISED_V2', 'Wrong protocol')
    require(m['seeds'] == SEEDS, 'Seeds changed')
    require([c['condition'] for c in m['conditions']] == CONDITIONS, 'Conditions changed')
    require(m['mia']['primary_attacks'] == PRIMARY, 'Primary attacks changed')
    require(m['mia']['new_shadow_models_trained'] == 0 and m['mia']['target_score_tuning'] is False, 'Attack protocol changed')
    require(all(v is True for v in m['protocol_gates'].values()), 'Source protocol gate failed')
    require(len(m['output_sha256']) == 15, 'Incomplete artifact index')
    require(set(blobs) == set(m['output_sha256']) | {'repeated_runs_manifest.json'}, 'Unexpected/missing artifact')
    for name, expected in m['output_sha256'].items():
        require(hashlib.sha256(blobs[name]).hexdigest() == expected, f'Checksum failed: {name}')
    cfg = json.loads(blobs['config.json'])
    for k,v in cfg.items():
        require(m.get(k) == v, f'Manifest/config mismatch: {k}')
    tables = {name: pd.read_csv(io.BytesIO(data)) for name,data in blobs.items() if name.endswith('.csv')}
    ids = tables['repeated_run_ids_results.csv']
    run = tables['repeated_run_configs.csv']
    require(len(ids)==45 and len(run)==15, 'Incomplete run tables')
    expected = {(c,s) for c in CONDITIONS for s in SEEDS}
    require(set(zip(run.condition,run.seed)) == expected and not run.duplicated(['condition','seed']).any(), 'Duplicate/missing run')
    require(run.model_sha256.nunique()==15, 'Model identities are not unique')
    source_by_seed = np.where(run.seed==42, 'accepted_experiment_05', 'trained_experiment_06')
    require((run.run_source == source_by_seed).all(), 'Wrong seed provenance')
    require(run.source_device.eq('cpu').all(), 'Mixed source devices')
    require(run.epochs.eq(30).all() and run.batch_size.eq(256).all() and run.optimizer.eq('Adam').all(), 'Training controls changed')
    close(run.learning_rate,.001,'Learning rate changed');close(run.weight_decay,.0001,'Weight decay changed')
    require((run.formal_dp == run.condition.ne('non_private')).all(), 'Formal DP labels incorrect')
    require(run.secure_mode.eq(False).all(), 'Secure-mode protocol changed')
    for c,e,noise in [('dp_eps_4',4,.6817626953125),('dp_eps_2',2,.88134765625)]:
        r=run[run.condition==c]
        close(r.target_epsilon,e,'Target epsilon mismatch')
        require((r.actual_epsilon-e).abs().le(.1).all(),'Actual epsilon outside accepted tolerance')
        close(r.delta,1/88181,'Delta changed',1e-15)
        close(r.noise_multiplier,noise,'Noise multiplier changed')
        close(r.max_grad_norm,1,'Clipping changed');close(r.sample_rate,1/345,'Sampling rate changed')
        require(r.accountant.eq('prv').all() and r.poisson_sampling.eq(True).all(),'Accountant/sampling mismatch')
    require(run[run.condition=='non_private'].actual_epsilon.isna().all(),'Non-private has finite epsilon')
    manifest_runs={(r['condition'],r['seed']):r for r in m['target_run_configs']}
    for _,r in run.iterrows():
        mr=manifest_runs[(r.condition,int(r.seed))]
        for k in ['model_sha256','cache_fingerprint','run_source']:
            require(r[k]==mr[k],f'Run manifest disagreement: {k}')
        sw=mr['source_software']
        require(sw['torch']=='2.11.0+cpu' and sw['opacus']=='1.6.0' and sw['numpy']=='2.1.3' and sw['pandas']=='2.2.3','Source software mismatch')
        require(sw.get('sklearn',sw.get('scikit-learn'))=='1.6.1','Source sklearn mismatch')
        for k in ['actual_epsilon','noise_multiplier','delta','selected_threshold']:
            close(r[k], np.nan if mr[k] is None else mr[k],f'Run manifest mismatch: {k}')
    require(not ids.duplicated(['condition','seed','split','threshold_policy']).any(),'Duplicate IDS evaluation')
    for _,r in ids.iterrows():
        tn,fp,fn,tp=[float(r[k]) for k in ['tn','fp','fn','tp']]
        require(all(x>=0 and x.is_integer() for x in [tn,fp,fn,tp]), 'Invalid confusion counts')
        den=tn+fp+fn+tp
        metrics={'recall':tp/(tp+fn),'fnr':fn/(tp+fn),'fpr':fp/(fp+tn),'precision':tp/(tp+fp) if tp+fp else 0,'f1':2*tp/(2*tp+fp+fn),'accuracy':(tp+tn)/den}
        for key,value in metrics.items():close(r[key],value,f'IDS count/metric mismatch: {key}')
        if r['split']=='KDDTest+':require(den==22544 and tp+fn==12833 and tn+fp==9711,'Test population changed')
        else:require(r['split']=='target_validation' and den==12597,'Validation population changed')
    tuned=ids[(ids['split']=='KDDTest+') & (ids.threshold_policy=='validation_selected_F2')].copy()
    require(len(tuned)==15 and set(zip(tuned.condition,tuned.seed))==expected,'Tuned IDS rows incomplete')
    for _,r in tuned.iterrows():
        rr=run[(run.condition==r.condition)&(run.seed==r.seed)].iloc[0]
        close(r.threshold,rr.selected_threshold,'Threshold/config mismatch')
    for policy,file in [('primary','repeated_run_mia_results.csv'),('secondary','secondary_mia_results.csv')]:
        d=tables[file]
        require(len(d)==30 and not d.duplicated(['condition','seed','threat_model']).any(),'Incomplete MIA table')
        require(set(zip(d.condition,d.seed,d.threat_model))=={(c,s,t) for c,s in expected for t in THREATS},'MIA key mismatch')
        require(d.analysis_role.eq(policy).all(),'Mixed attacker policy')
        require(d.members.eq(12597).all() and d.nonmembers.eq(12597).all() and d.n.eq(25194).all(),'MIA population changed')
        require(np.isfinite(d[MIA_METRICS]).all().all() and ((d[MIA_METRICS]>=0)&(d[MIA_METRICS]<=1)).all().all(),'Invalid MIA values')
        for (c,t),group in d.groupby(['condition','threat_model']):
            require(group.attack_model.nunique()==1 and group.operating_threshold.nunique()==1,'Attacker changed across seeds')
            if policy=='primary':require(group.attack_model.iloc[0]==PRIMARY[t],'Primary family changed')
    def check_summaries(summary, values_for):
        require(not summary.duplicated([c for c in ['condition','comparison_condition','domain','threat_model','metric'] if c in summary]).any(),'Duplicate summaries')
        for _,r in summary.iterrows():
            require(r.n_seeds==5,'Summary missing seeds')
            close(r[['mean','standard_deviation','ci_low','ci_high']],stats(values_for(r)),f'Summary mismatch: {r.metric}')
    primary=tables['repeated_run_mia_results.csv']
    def primary_values(r):
        data=tuned if r.domain=='IDS' else run if r.domain=='privacy_accounting' else primary
        mask=data.condition.eq(r.condition)
        if r.domain=='MIA':mask &= data.threat_model.eq(r.threat_model)
        return data.loc[mask,r.metric]
    require(len(tables['repeated_run_summary.csv'])==53,'Incomplete primary summary')
    check_summaries(tables['repeated_run_summary.csv'],primary_values)
    secondary=tables['secondary_mia_results.csv']
    require(len(tables['secondary_mia_summary.csv'])==30,'Incomplete secondary summary')
    check_summaries(tables['secondary_mia_summary.csv'],lambda r:secondary.loc[secondary.condition.eq(r.condition)&secondary.threat_model.eq(r.threat_model),r.metric])
    for policy,diffname,sumname in [('primary','repeated_run_paired_differences.csv','repeated_run_paired_summary.csv'),('secondary','secondary_mia_paired_differences.csv','secondary_mia_paired_summary.csv')]:
        diff=tables[diffname];summary=tables[sumname]
        require(len(diff)==(170 if policy=='primary' else 100),'Incomplete paired differences')
        require(len(summary)==(34 if policy=='primary' else 20),'Incomplete paired summary')
        for _,r in diff.iterrows():
            data=tuned if policy=='primary' and r.domain=='IDS' else primary if policy=='primary' else secondary
            mask=data.seed.eq(r.seed)
            if 'threat_model' in data:mask &= data.threat_model.eq(r.threat_model)
            a=data.loc[mask & data.condition.eq('non_private'),r.metric]
            b=data.loc[mask & data.condition.eq(r.comparison_condition),r.metric]
            require(len(a)==len(b)==1,'Unpaired observations')
            close(r.difference_dp_minus_non_private,b.iloc[0]-a.iloc[0],'Paired difference incorrect')
        def paired_values(r):
            mask=diff.comparison_condition.eq(r.comparison_condition)&diff.metric.eq(r.metric)
            if pd.notna(r.threat_model):mask &= diff.threat_model.eq(r.threat_model)
            if 'domain' in diff:mask &= diff.domain.eq(r.domain)
            d=diff.loc[mask]
            require(set(d.seed)==set(SEEDS) and not d.seed.duplicated().any(),'Paired seeds incorrect')
            return d.difference_dp_minus_non_private
        check_summaries(summary,paired_values)
    verification=tables['seed42_import_verification.csv']
    require(len(verification)==24 and verification.verification_passed.eq(True).all(),'Seed-42 verification incomplete')
    for _,r in verification.iterrows():
        tol=1e-8 if r.metric=='mia_auc' else 1e-10
        close(r.absolute_difference,abs(r.recomputed_value-r.accepted_experiment_05_value),'Incorrect saved verification difference')
        require(r.absolute_difference<=tol and r.verification_atol==tol,'Verification tolerance violated')
    fixed=tables['fixed_attacker_verification.csv']
    require(len(fixed)==12 and fixed.verification_passed.eq(True).all(),'Attacker reconstruction failed')
    close(fixed.accepted_operating_threshold,fixed.reconstructed_operating_threshold,'Calibration threshold mismatch',1e-8)
    close(fixed.accepted_shadow_calibration_auc,fixed.reconstructed_shadow_calibration_auc,'Calibration AUC mismatch',1e-10)
    sample=tables['target_mia_sample_manifest.csv']
    require(len(sample)==25194 and sample.membership.value_counts().to_dict()=={0:12597,1:12597},'MIA sample imbalance')
    require(sample.row_id.nunique()==25194,'Overlapping/duplicate MIA records')
    counts=sample.groupby(['membership','true_label']).size().unstack()
    require(counts.loc[0].equals(counts.loc[1]),'MIA class balance differs')
    report={'accepted_for_final_analysis':True,'manifest_sha256':hashlib.sha256(blobs['repeated_runs_manifest.json']).hexdigest(),'verified_artifact_checksums':15,'condition_seed_runs':15,'new_target_runs':12,'primary_mia_rows':30,'secondary_mia_rows':30,'summaries_and_paired_differences_recomputed':True,'neural_predictions_independently_recomputed':False,'source_model_files_available_in_bundle':False,'scope':'Training-seed stability conditional on fixed data and attackers; no universal optimum or leakage-reduction claim.'}
    return m,tables,report



In [4]:
blobs = read_bundle(evidence_source)
require(hashlib.sha256(blobs["repeated_runs_manifest.json"]).hexdigest() == ACCEPTED_MANIFEST_SHA256,
        "This is not the reviewed Experiment 06 bundle. Stop for review; do not change the accepted hash.")
manifest06, tables06, audit_report = audit_bundle(blobs)
print("Experiment 06 independent evidence gate: PASSED")
display(pd.DataFrame([audit_report]))


Experiment 06 independent evidence gate: PASSED


,accepted_for_final_analysis,manifest_sha256,verified_artifact_checksums,condition_seed_runs,new_target_runs,primary_mia_rows,secondary_mia_rows,summaries_and_paired_differences_recomputed,neural_predictions_independently_recomputed,source_model_files_available_in_bundle,scope
0,True,f1d93ba4362ebe9308fd7e47bcf111f90a9c8d2e8ebd7e...,15,15,12,30,30,True,False,False,Training-seed stability conditional on fixed d...


## 4. Load the frozen Experiment 05 context

Only ε≈8 uses single-seed context in the budget figures. It is never pooled with five-seed results.
The embedded CSV snapshots are from repository commit `19e099bd`; no web fetch or model rerun is needed.
Group diagnostics remain Experiment 05-only exploratory evidence, including the 208-record Rare group.


In [5]:
EXP05_SNAPSHOTS = {'dp_sgd_ids_results.csv': 'condition,formal_dp,actual_epsilon,delta,split,threshold_policy,threshold,accuracy,precision,recall,f1,fnr,fpr,roc_auc,pr_auc,tn,fp,fn,tp\nnon_private,False,,,target_validation,default_0_5,0.5,0.9948400412796696,0.9942038868053188,0.994712604468702,0.9944581805780544,0.0052873955312979,0.005049005049005,0.9998172305864614,0.9998000499766132,6700,34,31,5832\nnon_private,False,,,KDDTest+,default_0_5,0.5,0.8107256919801278,0.9635281385281386,0.6937582794358295,0.8066869025506275,0.3062417205641705,0.0347029142209865,0.9103638879025004,0.935713551393365,9374,337,3930,8903\nnon_private,False,,,KDDTest+,validation_selected_F2,0.29,0.8165365507452094,0.9589445910290236,0.7080183900880542,0.814595660749507,0.2919816099119457,0.0400576665636906,0.9103638879025004,0.935713551393365,9322,389,3747,9086\ndp_eps_8,True,7.99364894672444,1.134031140495118e-05,target_validation,default_0_5,0.5,0.98253552433119,0.9906103286384976,0.9716868497356302,0.9810573445841226,0.0283131502643697,0.008019008019008,0.9972359038963168,0.9974620472754692,6680,54,166,5697\ndp_eps_8,True,7.99364894672444,1.134031140495118e-05,KDDTest+,default_0_5,0.5,0.7366483321504613,0.915621986499518,0.5919114782202135,0.7190117847508164,0.4080885217797865,0.0720832046133251,0.8375410502780734,0.8967102529314104,9011,700,5237,7596\ndp_eps_8,True,7.99364894672444,1.134031140495118e-05,KDDTest+,validation_selected_F2,0.03,0.801277501774308,0.9201287596821244,0.7127717603054625,0.8032844471766049,0.2872282396945375,0.0817629492328287,0.8375410502780734,0.8967102529314104,8917,794,3686,9147\ndp_eps_4,True,3.9982670408726286,1.134031140495118e-05,target_validation,default_0_5,0.5,0.9817416845280622,0.9909360292835976,0.9696401159815794,0.9801724137931036,0.0303598840184206,0.0077220077220077,0.9970085692412148,0.9972780029493156,6682,52,178,5685\ndp_eps_4,True,3.9982670408726286,1.134031140495118e-05,KDDTest+,default_0_5,0.5,0.734652235628105,0.9152624560552794,0.5883269695316762,0.7162508300920216,0.4116730304683239,0.0719802286067346,0.83883158045028,0.8977608403572535,9012,699,5283,7550\ndp_eps_4,True,3.9982670408726286,1.134031140495118e-05,KDDTest+,validation_selected_F2,0.02,0.8066447835344216,0.9162900373354294,0.7267201745499883,0.8105688583720829,0.2732798254500116,0.0877355576150756,0.83883158045028,0.8977608403572535,8859,852,3507,9326\ndp_eps_2,True,1.999037532733395,1.134031140495118e-05,target_validation,default_0_5,0.5,0.980074620941494,0.9919354838709676,0.965034965034965,0.978300337166076,0.0349650349650349,0.0068310068310068,0.9967799175116248,0.9970128857462908,6688,46,205,5658\ndp_eps_2,True,1.999037532733395,1.134031140495118e-05,KDDTest+,default_0_5,0.5,0.7336763662171752,0.9148341635281254,0.5867684874931817,0.7149639194834789,0.4132315125068183,0.0721861806199155,0.8334451721934482,0.895967351762364,9010,701,5303,7530\ndp_eps_2,True,1.999037532733395,1.134031140495118e-05,KDDTest+,validation_selected_F2,0.03,0.7853530872959545,0.916875260742595,0.6850307800202603,0.7841755497078632,0.3149692199797397,0.0820718772526001,0.8334451721934482,0.895967351762364,8914,797,4042,8791\n\n', 'dp_sgd_mia_results.csv': 'condition,subset,threat_model,attack_model,feature_set,n,members,nonmembers,mia_auc,mia_auc_ci_low,mia_auc_ci_high,mia_advantage,mia_advantage_ci_low,mia_advantage_ci_high,mia_balanced_accuracy,mia_precision,mia_recall,tpr_at_1pct_fpr,tpr_at_5pct_fpr,operating_threshold,shadow_calibration_auc,formal_dp,target_epsilon,actual_epsilon,delta,noise_multiplier\nnon_private,overall,score_only_black_box,confidence_threshold,confidence,25194,12597,12597,0.5013959247880491,0.4943644966773788,0.5084411671475128,0.005636262602206843,0.002683153501243421,0.0189420163705695,0.5000396919901564,0.5000320286977131,0.6196713503215051,0.0,0.0,0.99998546,0.4994816978462719,False,,,,\nnon_private,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.501777124808777,0.4949262059877497,0.5089140573949186,0.008970389775343368,0.0029625350429185212,0.01974897487807678,0.5020242914979758,0.5017138248538208,0.5926014130348496,0.0,0.0,0.49985956900833967,0.5009336124463495,False,,,,\nnon_private,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.4951438518449535,0.48825018400020637,0.5015447578716631,0.0027784393109470534,0.0013422531375084682,0.012294365393296772,0.4971818686988966,0.4934706639691006,0.21298721917916966,0.01103437326347543,0.05032944351829801,0.6450300573765316,0.49814598581651875,False,,,,\nnon_private,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.5014386384991358,0.49426899055847767,0.5084872123249896,0.005636262602206843,0.003475770607452,0.018976777424295155,0.5000396919901564,0.5000320286977131,0.6196713503215051,0.0,0.0,-1.4543639e-05,0.49978941001571253,False,,,,\nnon_private,overall,label_aware_audit,logistic_regression,loss + correctness,25194,12597,12597,0.5014384368410132,0.4948464054862298,0.508629932014052,0.005636262602206843,0.0032240133110442632,0.01929770498253484,0.5000396919901564,0.5000320286977131,0.6196713503215051,0.0,0.0,0.5018095729987685,0.49977742491247756,False,,,,\nnon_private,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.49674818715176994,0.48966190039525403,0.5034998559761062,0.00214336746844479,0.0005608530022117483,0.015047335518105474,0.4997221560689053,0.4882154882154882,0.011510677145352068,0.009605461617845519,0.04667778042391046,0.8539037540613438,0.4965752031894037,False,,,,\ndp_eps_8,overall,score_only_black_box,confidence_threshold,confidence,25194,12597,12597,0.5020088299916093,0.49489358141990053,0.5088954288713027,0.011113757243788214,0.003238301521483249,0.023842633865105014,0.5033738191632928,0.5048017173200768,0.35468762403746923,0.0,0.0,0.9999957,0.4985556406779344,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375\ndp_eps_8,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.5028193034389409,0.4960491618611018,0.5092906990746413,0.012145748987854255,0.004590745805286778,0.024208304441614193,0.5000793839803127,0.5000798722044728,0.497023100738271,0.0,0.0,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375\ndp_eps_8,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.5000880426759506,0.49329197042558626,0.5069371296027803,0.007620862110026216,0.001677524640923006,0.021508388391060975,0.4988489322854648,0.49237243556023147,0.07430340557275542,0.009605461617845519,0.04937683575454473,0.7176487650728317,0.5000288285211884,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375\ndp_eps_8,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.5020314314558779,0.4949218687014085,0.5085837468937429,0.011113757243788214,0.0031873098127327792,0.023426220989092823,0.5005953798523458,0.5003941352672238,0.7558942605382234,0.0,0.0,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375\ndp_eps_8,overall,label_aware_audit,logistic_regression,loss + correctness,25194,12597,12597,0.5020314314558779,0.49464095534072955,0.5099240057275198,0.011113757243788214,0.00377230991309867,0.0230136660799044,0.5005953798523458,0.5003941352672238,0.7558942605382234,0.0,0.0,0.5004377707589382,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375\ndp_eps_8,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.4997120605591442,0.49245529345298233,0.5066509646437056,0.008891005795030549,0.0018699025661804724,0.020429399684228376,0.5004366118917203,0.5002275642351773,0.9597523219814241,0.009684845598158291,0.048027308089227595,0.23224901910670478,0.49399995611763464,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375\ndp_eps_4,overall,score_only_black_box,confidence_threshold,confidence,25194,12597,12597,0.5016463652708314,0.4949431249510033,0.5086330883104653,0.012225132968167018,0.0028045560114464715,0.024176830519810566,0.5044455028975152,0.5060606060606061,0.371199491942526,0.0,0.0,0.9999925,0.4977482624971514,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125\ndp_eps_4,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.5028840672053674,0.4960046813501511,0.5101362495917817,0.01286020481066924,0.004840795558133028,0.02613562033037513,0.500515995872033,0.5005141998259631,0.502262443438914,0.0,0.0,0.49985580787381517,0.501753946133048,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125\ndp_eps_4,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.5030609465861262,0.49601427880026816,0.510202185389338,0.01254266888941813,0.005715593541725028,0.024731594093806476,0.5001587679606255,0.5007541478129713,0.10542192585536239,0.009287925696594427,0.05072636341986187,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125\ndp_eps_4,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.50166603008869,0.4939076831350602,0.508831406210083,0.012225132968167018,0.003889569926096348,0.02375812781505225,0.5043661189172025,0.5059536696254601,0.37104072398190047,0.0,0.0,-7.5102134e-06,0.49811670044991274,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125\ndp_eps_4,overall,label_aware_audit,logistic_regression,loss + correctness,25194,12597,12597,0.50166603008869,0.494649528629854,0.5086196019079819,0.012225132968167018,0.003403620597723435,0.024500300523009343,0.5043661189172025,0.5059536696254601,0.37104072398190047,0.0,0.0,0.5004500829699141,0.49811670044991274,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125\ndp_eps_4,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.5035969507250079,0.49690527093004516,0.5115131108743155,0.013415892672858698,0.004949357408164052,0.02764709825871214,0.5005556878621894,0.5012114918656975,0.22989600698579027,0.008573469873779471,0.04707470032547432,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125\ndp_eps_2,overall,score_only_black_box,confidence_threshold,confidence,25194,12597,12597,0.5016036547106528,0.4940641420443254,0.5087106082190275,0.011272525204413797,0.00325088324699655,0.023440474019334332,0.5033341271731364,0.5043677204658902,0.38501230451694846,0.0,0.0,0.9999832,0.49679576185577184,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625\ndp_eps_2,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.5029159922068966,0.49599576095640313,0.5100713757478378,0.01174882908629038,0.004590214416136715,0.02426482583062481,0.5008732237834405,0.5008639648130694,0.5062316424545527,0.0,0.0,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625\ndp_eps_2,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.4951362644580918,0.488118542447519,0.5019215204178088,0.0034928951337620112,0.0021932220248332903,0.012672001301678453,0.5002778439310948,0.5016595542911333,0.08398825117091371,0.012145748987854251,0.05088513138048742,0.712645603849439,0.4973062440762115,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625\ndp_eps_2,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.5016083243565534,0.493911662855753,0.5084787963587285,0.011193141224100978,0.0029342648858812007,0.024212985405050097,0.5032547431928237,0.504264614104431,0.38485353655632293,0.0,0.0,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625\ndp_eps_2,overall,label_aware_audit,logistic_regression,loss + correctness,25194,12597,12597,0.5016083243565534,0.49458416191161186,0.5080185135649493,0.011193141224100978,0.003817825979567842,0.023404196533699993,0.5032547431928237,0.504264614104431,0.38485353655632293,0.0,0.0,0.5004034382011459,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625\ndp_eps_2,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.49335113004075903,0.4863216392579739,0.5002401455996364,0.003254743192823688,0.002102689173272209,0.011002841588986264,0.4990077002460903,0.4970442184913691,0.1668651266174486,0.01182821306660316,0.05048821147892355,0.6149691904224103,0.49427559773875,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625\n\n', 'dp_sgd_group_analysis.csv': 'condition,subset,threat_model,attack_model,feature_set,n,members,nonmembers,mia_auc,mia_auc_ci_low,mia_auc_ci_high,mia_advantage,mia_advantage_ci_low,mia_advantage_ci_high,mia_balanced_accuracy,mia_precision,mia_recall,tpr_at_1pct_fpr,tpr_at_5pct_fpr,operating_threshold,shadow_calibration_auc,formal_dp,target_epsilon,actual_epsilon,delta\nnon_private,binary_group=Attack,score_only_black_box,logistic_regression,prob_attack,11726,5863,5863,0.5045752889277598,0.49530801181008094,0.5137104938882707,0.01210984137813409,0.005693362522219405,0.029122043668788553,0.5,0.5,1.0,0.0,0.0,0.49985956900833967,0.5009336124463495,False,,,\nnon_private,binary_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.5026672359980943,0.4932788971024487,0.5120508847358834,0.01678051678051684,0.006204595930734623,0.03239984309916739,0.5037867537867537,0.5080875356803045,0.2378972378972379,0.00891000891000891,0.049005049005049005,0.49985956900833967,0.5009336124463495,False,,,\nnon_private,family_group=DoS,score_only_black_box,logistic_regression,prob_attack,9186,4593,4593,0.502973575261894,0.49481314740008436,0.5116248116080763,0.007620291748312691,0.00103566368206853,0.023447184608525927,0.5,0.5,1.0,0.0,0.0,0.49985956900833967,0.5009336124463495,False,,,\nnon_private,family_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.5026672359980943,0.49245725659707545,0.5125922047452088,0.01678051678051684,0.0075281068456148235,0.03228507322810031,0.5037867537867537,0.5080875356803045,0.2378972378972379,0.00891000891000891,0.049005049005049005,0.49985956900833967,0.5009336124463495,False,,,\nnon_private,family_group=Probe,score_only_black_box,logistic_regression,prob_attack,2332,1166,1166,0.5167039827708457,0.49473579893370045,0.5401913472461837,0.036020583190394584,0.020231383479206718,0.07994072125510499,0.5,0.5,1.0,0.0,0.0,0.49985956900833967,0.5009336124463495,False,,,\nnon_private,family_group=Rare,score_only_black_box,logistic_regression,prob_attack,208,104,104,0.6462185650887574,0.5675679432112418,0.72282929896603,0.24038461538461542,0.1634131928272007,0.38626171212821697,0.5,0.5,1.0,0.08653846153846154,0.18269230769230768,0.49985956900833967,0.5009336124463495,False,,,\nnon_private,binary_group=Attack,label_aware_audit,loss_threshold,loss,11726,5863,5863,0.5045752889277598,0.49569974534535755,0.5135117895168311,0.01210984137813409,0.006107362212361206,0.029029976494881107,0.5000852805730854,0.5000511404316252,0.8338734436295412,0.0,0.0,-1.4543639e-05,0.49978941001571253,False,,,\nnon_private,binary_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.49737626215202696,0.48791825940980504,0.5072286627543726,0.00579150579150578,0.0022789584938620404,0.024675594606366123,0.5,0.5,0.43317493317493316,0.0,0.03267003267003267,-1.4543639e-05,0.49978941001571253,False,,,\nnon_private,family_group=DoS,label_aware_audit,loss_threshold,loss,9186,4593,4593,0.502973575261894,0.49463239158646066,0.5118485321258163,0.007620291748312691,0.0013009036742834517,0.02294714829405484,0.4996734160679295,0.49982716902868995,0.9444807315480078,0.0,0.0,-1.4543639e-05,0.49978941001571253,False,,,\nnon_private,family_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.49737626215202696,0.4871152110431632,0.5072086492942925,0.00579150579150578,0.0022980330977111995,0.025944308367255507,0.5,0.5,0.43317493317493316,0.0,0.03267003267003267,-1.4543639e-05,0.49978941001571253,False,,,\nnon_private,family_group=Probe,label_aware_audit,loss_threshold,loss,2332,1166,1166,0.5167039827708457,0.4952565310023664,0.5394391989616116,0.036020583190394584,0.022255564402540697,0.08395546970092843,0.5017152658662093,0.5018214936247724,0.4725557461406518,0.0,0.0,-1.4543639e-05,0.49978941001571253,False,,,\nnon_private,family_group=Rare,label_aware_audit,loss_threshold,loss,208,104,104,0.6462185650887574,0.5712081517219127,0.7165726288945093,0.24038461538461542,0.16004385307346328,0.3807012992373817,0.5,0.0,0.0,0.08653846153846154,0.18269230769230768,-1.4543639e-05,0.49978941001571253,False,,,\ndp_eps_8,binary_group=Attack,score_only_black_box,logistic_regression,prob_attack,11726,5863,5863,0.5086077960262075,0.4987914932495318,0.5182648001490167,0.025754733071806202,0.011753681957703216,0.04424479304143042,0.500767525157769,0.5003877638948729,0.9904485758144295,0.0,0.0,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,binary_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.5026273544242061,0.49337878098423443,0.5125403911456557,0.012474012474012475,0.00673963601219855,0.027801519552480388,0.4994802494802495,0.4961748633879781,0.06741906741906742,0.011286011286011286,0.05049005049005049,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=DoS,score_only_black_box,logistic_regression,prob_attack,9186,4593,4593,0.5089706886950953,0.49888943698111965,0.5188383685187211,0.028957108643588025,0.01456756114748732,0.04791092029041215,0.5001088613106902,0.5000549269471602,0.9910733725234052,0.0,0.0,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.5026273544242061,0.49246984605205246,0.5122336856217572,0.012474012474012475,0.0064352907375298,0.02830855619811784,0.4994802494802495,0.4961748633879781,0.06741906741906742,0.011286011286011286,0.05049005049005049,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=Probe,score_only_black_box,logistic_regression,prob_attack,2332,1166,1166,0.501962405373519,0.47972129341103115,0.5252554742827278,0.023156089193825058,0.010903504923331064,0.06684387851397147,0.5038593481989708,0.5019472090004327,0.9948542024013722,0.0,0.0,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=Rare,score_only_black_box,logistic_regression,prob_attack,208,104,104,0.5671227810650887,0.49262385531135533,0.641761653229543,0.15384615384615385,0.07472459612775928,0.2961403474116708,0.4951923076923077,0.4973821989528796,0.9134615384615384,0.028846153846153848,0.08653846153846154,0.4998898396475398,0.5013578359506039,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,binary_group=Attack,label_aware_audit,loss_threshold,loss,11726,5863,5863,0.5086077960262075,0.4989019230106676,0.5177291403044234,0.025754733071806202,0.010854108650250225,0.044293236037778794,0.5004264028654273,0.500232450023245,0.9176189663994542,0.0,0.0,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,binary_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.4973719509285295,0.4870490147505621,0.50663463690545,0.006534006534006542,0.0029587689529551394,0.024542180116310677,0.5007425007425008,0.5006043026347595,0.6150876150876151,0.011137511137511137,0.0444015444015444,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=DoS,label_aware_audit,loss_threshold,loss,9186,4593,4593,0.5089706886950953,0.4988363858843292,0.5194502306940012,0.028957108643588025,0.015204476148111915,0.048619480118886,0.5015240583496625,0.5007959972708665,0.9588504245591117,0.0,0.0,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.4973719509285295,0.487968919595889,0.5068553122417081,0.006534006534006542,0.0029648097313702826,0.02493634368763072,0.5007425007425008,0.5006043026347595,0.6150876150876151,0.011137511137511137,0.0444015444015444,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=Probe,label_aware_audit,loss_threshold,loss,2332,1166,1166,0.501962405373519,0.47722333600852457,0.5249182535818466,0.023156089193825058,0.012062545291100027,0.06861908687235471,0.4961406518010292,0.49770525242223357,0.8370497427101201,0.0,0.0,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_8,family_group=Rare,label_aware_audit,loss_threshold,loss,208,104,104,0.5671227810650887,0.4879503779395296,0.6440644981929387,0.15384615384615385,0.07364866236963515,0.29896566678654035,0.5,0.0,0.0,0.028846153846153848,0.08653846153846154,-0.0008858789,0.49895824436662656,True,8.0,7.99364894672444,1.134031140495118e-05\ndp_eps_4,binary_group=Attack,score_only_black_box,random_forest,prob_attack,11726,5863,5863,0.5025692099923639,0.4932996232580952,0.5116008641205183,0.018250042640286512,0.00639816460826508,0.035526754691052084,0.500767525157769,0.5081669691470054,0.047757120927852635,0.012792085962817671,0.050827221558928874,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,binary_group=Normal,score_only_black_box,random_forest,prob_attack,13468,6734,6734,0.5041024654553018,0.4946251203932612,0.5141706842915658,0.013068013068013085,0.006040920501355545,0.028674714855515246,0.4996287496287496,0.4988100904331271,0.15562815562815563,0.008019008019008018,0.04544104544104544,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=DoS,score_only_black_box,random_forest,prob_attack,9186,4593,4593,0.5019153949707829,0.49166670065338924,0.5117760045629386,0.016546919224907475,0.0053045150632954376,0.033754612246961,0.5,0.5,0.025908991944263007,0.011757021554539516,0.047028086218158065,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=Normal,score_only_black_box,random_forest,prob_attack,13468,6734,6734,0.5041024654553018,0.4949291264538845,0.5142147330370374,0.013068013068013085,0.006653776696136662,0.02898260084035873,0.4996287496287496,0.4988100904331271,0.15562815562815563,0.008019008019008018,0.04544104544104544,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=Probe,score_only_black_box,random_forest,prob_attack,2332,1166,1166,0.5058397741615646,0.48399906987550523,0.5293528446891144,0.0317324185248713,0.013601398897555652,0.07219573887884735,0.5042881646655232,0.518796992481203,0.1183533447684391,0.010291595197255575,0.05317324185248713,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=Rare,score_only_black_box,random_forest,prob_attack,208,104,104,0.5235299556213017,0.44053596407227985,0.6050603736640288,0.15384615384615385,0.06991417757233509,0.26983279846461894,0.4951923076923077,0.48936170212765956,0.22115384615384615,0.0,0.028846153846153848,0.6580489320797822,0.5027942052038461,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,binary_group=Attack,label_aware_audit,random_forest,loss + correctness,11726,5863,5863,0.5046393329944996,0.49545374119713975,0.5135856317010601,0.02012621524816649,0.009411740597311286,0.03654768534941177,0.495650690772642,0.4804297774366846,0.10677127750298482,0.011086474501108648,0.04912161009721985,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,binary_group=Normal,label_aware_audit,random_forest,loss + correctness,13468,6734,6734,0.5005336544911835,0.49074159366600895,0.5102036095887802,0.011880011880011865,0.00283445151002651,0.030520432354522276,0.5048262548262548,0.5072625698324023,0.3370953370953371,0.006534006534006534,0.04425304425304425,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=DoS,label_aware_audit,random_forest,loss + correctness,9186,4593,4593,0.5015066139941937,0.4917019463204956,0.5109760124899921,0.014369693011103912,0.0078059785667909274,0.03256899433751024,0.49586327019377313,0.4699367088607595,0.06466361854996734,0.009362072719355541,0.0428913564119312,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=Normal,label_aware_audit,random_forest,loss + correctness,13468,6734,6734,0.5005336544911835,0.490978014396227,0.5102368169608255,0.011880011880011865,0.0029802596745008183,0.029440190773599818,0.5048262548262548,0.5072625698324023,0.3370953370953371,0.006534006534006534,0.04425304425304425,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=Probe,label_aware_audit,random_forest,loss + correctness,2332,1166,1166,0.5110010179793991,0.4877766113612339,0.5332064136117758,0.03945111492281306,0.013820701749720134,0.07910287881320338,0.49313893653516294,0.48653198653198654,0.2478559176672384,0.012864493996569469,0.06260720411663807,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_4,family_group=Rare,label_aware_audit,random_forest,loss + correctness,208,104,104,0.5471523668639053,0.46285808404231354,0.6253919907190717,0.11538461538461542,0.05392380914717917,0.2619337613946371,0.514423076923077,0.5194805194805194,0.38461538461538464,0.038461538461538464,0.07692307692307693,0.5588558222769949,0.5025996363157601,True,4.0,3.9982670408726286,1.134031140495118e-05\ndp_eps_2,binary_group=Attack,score_only_black_box,logistic_regression,prob_attack,11726,5863,5863,0.5082843174887954,0.4988776015384269,0.5188413389125426,0.024901927340951646,0.010863683051438266,0.043061817353360725,0.500767525157769,0.5003875635173543,0.9909602592529422,0.0,0.0,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,binary_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.5032389748011963,0.4935628518540976,0.5134647979362841,0.015295515295515288,0.007846094419967815,0.032229609555798906,0.500965250965251,0.5057983942908117,0.0841995841995842,0.012474012474012475,0.04855954855954856,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=DoS,score_only_black_box,logistic_regression,prob_attack,9186,4593,4593,0.5080236687669576,0.49669096484992625,0.5186379382445042,0.026344437187023728,0.013388655228507204,0.04662807253171163,0.49989113868930984,0.4999450609823096,0.9906379272806445,0.0,0.0,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.5032389748011963,0.4936993457765477,0.5134271850657998,0.015295515295515288,0.007620402386925849,0.031082068120971133,0.500965250965251,0.5057983942908117,0.0841995841995842,0.012474012474012475,0.04855954855954856,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=Probe,score_only_black_box,logistic_regression,prob_attack,2332,1166,1166,0.5038519928564914,0.4796060063281274,0.5267995357869235,0.02572898799313894,0.012252610635802621,0.06738896392068813,0.5030017152658662,0.501511879049676,0.9957118353344768,0.0,0.0,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=Rare,score_only_black_box,logistic_regression,prob_attack,208,104,104,0.5646264792899408,0.4876054119173385,0.6404090955785667,0.14423076923076927,0.07811355311355311,0.29903127416444714,0.5144230769230769,0.5076923076923077,0.9519230769230769,0.028846153846153848,0.07692307692307693,0.49980178014216076,0.5018781072656153,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,binary_group=Attack,label_aware_audit,loss_threshold,loss,11726,5863,5863,0.5082843174887954,0.49823316502571674,0.5179803691647406,0.024901927340951646,0.011398177154621448,0.044752241665521504,0.5108306327818523,0.5076423155614394,0.7194269145488658,0.0,0.0,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,binary_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.4967613780355093,0.48708449962718975,0.5064238195703409,0.00757350757350761,0.0021829062805233993,0.02627113484522107,0.49665874665874665,0.4827586206896552,0.09355509355509356,0.01098901098901099,0.04558954558954559,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=DoS,label_aware_audit,loss_threshold,loss,9186,4593,4593,0.5080236687669576,0.4977372495312459,0.5180594645665334,0.026344437187023728,0.012062465148386362,0.04824974446811473,0.511430437622469,0.5070422535211268,0.8229915088177662,0.0,0.0,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.4967613780355093,0.4874508423138957,0.5064878557174264,0.00757350757350761,0.002609398486132958,0.024809383098192663,0.49665874665874665,0.4827586206896552,0.09355509355509356,0.01098901098901099,0.04558954558954559,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=Probe,label_aware_audit,loss_threshold,loss,2332,1166,1166,0.5038519928564914,0.4789462297189927,0.5273637302872147,0.02572898799313894,0.012621503297828462,0.06870958284281913,0.5094339622641509,0.5128805620608899,0.37564322469982847,0.0,0.0,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05\ndp_eps_2,family_group=Rare,label_aware_audit,loss_threshold,loss,208,104,104,0.5646264792899408,0.49037771663415997,0.6404046499643066,0.14423076923076927,0.07739529914529916,0.31136602568367827,0.5,0.0,0.0,0.028846153846153848,0.07692307692307693,-1.6808652e-05,0.49704851394612953,True,2.0,1.9990375327333947,1.134031140495118e-05\n\n', 'dp_sgd_configs.csv': 'condition,formal_dp,target_epsilon,actual_epsilon,delta,noise_multiplier,max_grad_norm,batch_size,sample_rate,epochs,optimizer,learning_rate,weight_decay,accountant,poisson_sampling,secure_mode,training_seconds,selected_threshold,model_state_path,model_sha256\nnon_private,False,,,,,,256,,30,Adam,0.001,0.0001,,False,False,41.0140745639801,0.29000000000000004,/content/drive/MyDrive/ML-DP-NID/artifacts/models/dp_sgd_sweep/non_private.pt,c380553509ec2b66f3d8734a12905688e9a9d7178d420790c26dc1b869710102\ndp_eps_8,True,8.0,7.99364894672444,1.134031140495118e-05,0.560302734375,1.0,256,0.002898550724637681,30,Adam,0.001,0.0001,prv,True,False,91.87378263473511,0.03,/content/drive/MyDrive/ML-DP-NID/artifacts/models/dp_sgd_sweep/dp_eps_8.pt,affc8ebbbed875a98295e7c169c33211961f46e401405758b202a4e6b5cffc6b\ndp_eps_4,True,4.0,3.9982670408726286,1.134031140495118e-05,0.6817626953125,1.0,256,0.002898550724637681,30,Adam,0.001,0.0001,prv,True,False,85.54690051078796,0.02,/content/drive/MyDrive/ML-DP-NID/artifacts/models/dp_sgd_sweep/dp_eps_4.pt,7ca77b11bb5dc47f6f0549b6ef77019c2061916bd3be77ca8be9d81f4df92fcf\ndp_eps_2,True,2.0,1.9990375327333947,1.134031140495118e-05,0.88134765625,1.0,256,0.002898550724637681,30,Adam,0.001,0.0001,prv,True,False,85.47071266174316,0.03,/content/drive/MyDrive/ML-DP-NID/artifacts/models/dp_sgd_sweep/dp_eps_2.pt,1757fc45b5e89df8d8d46f84a981ec1319d949018c5542133799e2bee3baa26f\n\n'}
exp05 = {name: pd.read_csv(io.StringIO(text)) for name, text in EXP05_SNAPSHOTS.items()}
exp05_snapshot_hashes = {name: hashlib.sha256(text.encode()).hexdigest() for name,text in EXP05_SNAPSHOTS.items()}
# Verify that reused seed 42 is the same utility evidence as the frozen sweep.
ids = tables06["repeated_run_ids_results.csv"].copy()
for _, row in ids[ids.seed.eq(42)].iterrows():
    old = exp05["dp_sgd_ids_results.csv"]
    match = old[old.condition.eq(row.condition) & old["split"].eq(row["split"]) & old.threshold_policy.eq(row.threshold_policy)]
    require(len(match)==1, "Experiment 05 context mismatch")
    close(row[["recall","fnr","f1","fpr","precision","pr_auc"]], match.iloc[0][["recall","fnr","f1","fpr","precision","pr_auc"]], "Seed-42 utility differs from frozen sweep")


## 5. Produce final tables and preserve primary/secondary policies


In [6]:
exports = []
def save_table(frame, name, folder=TABLE_DIR):
    path = folder / name
    frame.to_csv(path, index=False)
    if path not in exports: exports.append(path)
    return path

def save_text(text, name):
    path = OUTPUT_DIR / name
    path.write_text(text, encoding="utf-8")
    if path not in exports: exports.append(path)
    return path

summary = tables06["repeated_run_summary.csv"]
paired = tables06["repeated_run_paired_summary.csv"]
secondary_summary = tables06["secondary_mia_summary.csv"]
configs = tables06["repeated_run_configs.csv"]
tuned = ids[ids["split"].eq("KDDTest+") & ids.threshold_policy.eq("validation_selected_F2")].copy()
labels = {"non_private":"Non-private", "dp_eps_4":"DP ε≈4", "dp_eps_2":"DP ε≈2", "dp_eps_8":"DP ε≈8 (single seed)"}
colors = {"non_private":"#333333", "dp_eps_4":"#087e8b", "dp_eps_2":"#c25a24", "dp_eps_8":"#8064a2"}

def summary_row(condition, metric, threat=None, policy="primary"):
    data = secondary_summary if policy=="secondary" else summary
    mask = data.condition.eq(condition) & data.metric.eq(metric)
    if threat is not None:
        mask &= data.threat_model.eq(threat)
    rows = data[mask]
    require(len(rows)==1, f"Ambiguous summary: {condition}, {metric}, {threat}")
    return rows.iloc[0]

def final_table(policy):
    records=[]
    for condition in CONDITIONS:
        private=condition!="non_private"
        for threat in THREATS:
            row={"condition":condition,"formal_dp":private,
                 "actual_epsilon":float(configs[configs.condition.eq(condition)].actual_epsilon.mean()) if private else None,
                 "delta":manifest06["dp"]["target_delta"] if private else None,
                 "n_seeds":5,"analysis_role":policy,"threat_model":threat}
            for metric in IDS_METRICS:
                r=summary_row(condition,metric)
                for stat in ["mean","standard_deviation","ci_low","ci_high"]:row[f"ids_{metric}_{stat}"]=r[stat]
            for metric in MIA_METRICS:
                r=summary_row(condition,metric,threat,policy)
                for stat in ["mean","standard_deviation","ci_low","ci_high"]:row[f"{metric}_{stat}"]=r[stat]
            records.append(row)
    return pd.DataFrame(records)

primary_final=final_table("primary")
secondary_final=final_table("secondary")
save_table(primary_final,"final_primary_privacy_utility.csv")
save_table(secondary_final,"final_secondary_privacy_utility.csv")
save_table(summary,"five_seed_summary.csv")
save_table(paired,"primary_paired_differences_summary.csv")
save_table(tables06["secondary_mia_paired_summary.csv"],"secondary_paired_differences_summary.csv")
save_table(ids,"ids_default_and_tuned_per_seed.csv")
save_table(configs,"privacy_accounting_per_seed.csv")
save_table(tables06["repeated_run_mia_results.csv"],"primary_mia_per_seed.csv")
save_table(tables06["secondary_mia_results.csv"],"secondary_mia_per_seed.csv")
save_table(exp05["dp_sgd_group_analysis.csv"].assign(evidence_scope="Experiment 05 seed 42; exploratory"),"exploratory_groups_single_seed.csv")
# Negative lower t bounds and zero-width intervals are retained in raw tables.
# They are not valid negative probabilities or claims of zero population uncertainty.
save_table(summary[summary.domain.ne("privacy_accounting") & ((summary.ci_low<0)|(summary.ci_high>1))].copy(),"unbounded_probability_t_intervals.csv")
save_text(json.dumps(audit_report,indent=2),"experiment06_acceptance_audit.json")
display(primary_final[["condition","actual_epsilon","threat_model","ids_recall_mean","ids_fpr_mean","ids_f1_mean","mia_auc_mean"]])


,condition,actual_epsilon,threat_model,ids_recall_mean,ids_fpr_mean,ids_f1_mean,mia_auc_mean
0,non_private,NaN,score_only_black_box,0.709343,0.056101,0.809750,0.502297
1,non_private,NaN,label_aware_audit,0.709343,0.056101,0.809750,0.500882
2,dp_eps_4,3.998267,score_only_black_box,0.713380,0.087550,0.801657,0.503009
3,dp_eps_4,3.998267,label_aware_audit,0.713380,0.087550,0.801657,0.501594
4,dp_eps_2,1.999038,score_only_black_box,0.701083,0.083822,0.794605,0.503268
5,dp_eps_2,1.999038,label_aware_audit,0.701083,0.083822,0.794605,0.501463


## 6. Budget figures with evidence strength shown explicitly

Solid points/bands: five-seed means and approximate 95% t intervals conditional on the fixed
sample and attackers. Hollow ε≈8 diamond: Experiment 05 seed 42, without a repeated-run interval.
The non-private reference is a separate horizontal band; it is never assigned a finite epsilon.
`pr_auc` means average precision, not trapezoidal area under the precision–recall curve.


In [7]:
figure_records=[]
def save_figure(fig,stem,caption):
    for extension in ["png","pdf"]:
        path=FIGURE_DIR/f"{stem}.{extension}"
        fig.savefig(path,bbox_inches="tight")
        if path not in exports: exports.append(path)
    figure_records[:] = [r for r in figure_records if r["figure"] != stem]
    figure_records.append({"figure":stem,"caption":caption})
    plt.close(fig)

def epsilon_value(condition):
    return float(configs[configs.condition.eq(condition)].actual_epsilon.mean())

def budget_axes(ax,metric,threat=None):
    for condition in ["dp_eps_2","dp_eps_4"]:
        r=summary_row(condition,metric,threat)
        ax.errorbar(epsilon_value(condition),r["mean"],
            yerr=[[max(0,r["mean"]-r.ci_low)],[max(0,r.ci_high-r["mean"])]],
            fmt="o",capsize=4,color=colors[condition],label=labels[condition]+"; 5 seeds")
    ref=summary_row("non_private",metric,threat)
    ax.axhline(ref["mean"],color=colors["non_private"],label="Non-private; 5 seeds")
    ax.axhspan(ref.ci_low,ref.ci_high,color=colors["non_private"],alpha=.09)
    if threat is None:
        data=exp05["dp_sgd_ids_results.csv"]
        old=data[data.condition.eq("dp_eps_8") & data["split"].eq("KDDTest+") & data.threshold_policy.eq("validation_selected_F2")].iloc[0]
    else:
        data=exp05["dp_sgd_mia_results.csv"]
        old=data[data.condition.eq("dp_eps_8") & data.threat_model.eq(threat) & data.attack_model.eq(PRIMARY[threat]) & data.subset.eq("overall")].iloc[0]
    eps8=float(exp05["dp_sgd_configs.csv"].query("condition == 'dp_eps_8'").actual_epsilon.iloc[0])
    ax.plot(eps8,old[metric],marker="D",markerfacecolor="none",markeredgecolor=colors["dp_eps_8"],linestyle="none",label="ε≈8; 1 seed, no interval")
    ax.set_xlabel("Actual ε (δ fixed at 1/88,181)")
    ax.grid(alpha=.2)
    ax.set_xticks([2,4,8])

for metric,title,stem in [("f1","IDS F1","01_f1_vs_epsilon"),("recall","IDS Recall","02_recall_vs_epsilon"),("fnr","IDS False Negative Rate","03_fnr_vs_epsilon")]:
    fig,ax=plt.subplots(figsize=(7,4.5));budget_axes(ax,metric)
    ax.set_ylabel(title);ax.set_title(title+" versus privacy budget")
    ax.legend(fontsize=8)
    save_figure(fig,stem,"Validation-selected F2 thresholds. Five-seed t intervals; ε≈8 is single-seed context.")
for metric,title,stem in [("mia_auc","MIA ROC-AUC","04_mia_auc_vs_epsilon"),("mia_advantage","MIA advantage","05_mia_advantage_vs_epsilon")]:
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    for ax,threat in zip(axes,THREATS):
        budget_axes(ax,metric,threat);ax.set_title(threat.replace("_"," "))
        ax.set_ylabel(title)
        if metric=="mia_auc":ax.axhline(.5,color="grey",linestyle=":",label="AUC = 0.5")
    axes[0].legend(fontsize=7)
    fig.tight_layout()
    save_figure(fig,stem,"Fixed primary attacks; conditional five-seed t intervals. Near-chance magnitude is not proof of no leakage.")
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,metric,title in zip(axes,["fpr","pr_auc"],["False Positive Rate","Average precision"]):
    budget_axes(ax,metric);ax.set_ylabel(title);ax.set_title(title)
axes[0].legend(fontsize=7);fig.tight_layout()
save_figure(fig,"10_false_alarms_and_average_precision","Additional IDS diagnostics prevent selecting a configuration using recall alone.")
save_table(summary,"budget_figure_five_seed_data.csv",PLOT_DIR)
save_table(exp05["dp_sgd_ids_results.csv"].query("condition == 'dp_eps_8'"),"epsilon8_single_seed_ids.csv",PLOT_DIR)
save_table(exp05["dp_sgd_mia_results.csv"].query("condition == 'dp_eps_8'"),"epsilon8_single_seed_mia.csv",PLOT_DIR)


PosixPath('/content/drive/MyDrive/ML-DP-NID/results/final_analysis/plot_data/epsilon8_single_seed_mia.csv')

## 7. Joint privacy–utility plots and descriptive frontiers

These scatter plots compare observed means and show individual seeds. They do not optimize a
weighted objective or declare a deployment-ready setting. The utility frontier uses Recall/FPR
only; formal epsilon is deliberately not interpreted as interchangeable with empirical MIA AUC.


In [8]:
primary_mia=tables06["repeated_run_mia_results.csv"]
joint=tuned.merge(primary_mia,on=["condition","seed"],suffixes=("_ids","_mia"),validate="one_to_many")
save_table(joint,"joint_utility_mia_per_seed.csv",PLOT_DIR)
for metric,title,stem in [("f1","IDS F1","06_f1_vs_mia_auc"),("fnr","IDS FNR","07_fnr_vs_mia_auc")]:
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    for ax,threat in zip(axes,THREATS):
        for condition in CONDITIONS:
            r=joint[joint.condition.eq(condition)&joint.threat_model.eq(threat)]
            ax.scatter(r.mia_auc,r[metric],color=colors[condition],s=22,alpha=.35)
            x=summary_row(condition,"mia_auc",threat);y=summary_row(condition,metric)
            ax.errorbar(x["mean"],y["mean"],xerr=[[x["mean"]-x.ci_low],[x.ci_high-x["mean"]]],
                yerr=[[y["mean"]-y.ci_low],[y.ci_high-y["mean"]]],fmt="o",capsize=3,color=colors[condition],label=labels[condition])
        ax.axvline(.5,color="grey",ls=":");ax.set_xlabel("MIA ROC-AUC");ax.set_ylabel(title)
        ax.set_title(threat.replace("_"," "));ax.grid(alpha=.2)
    axes[0].legend(fontsize=8);fig.tight_layout()
    save_figure(fig,stem,"Small points are target seeds. Error bars are marginal conditional t intervals, not a joint confidence region.")

frontier=[]
for condition in CONDITIONS:
    frontier.append({"condition":condition,"epsilon":epsilon_value(condition) if condition!='non_private' else None,
        "recall":summary_row(condition,"recall")["mean"],"fpr":summary_row(condition,"fpr")["mean"],
        "f1":summary_row(condition,"f1")["mean"]})
frontier=pd.DataFrame(frontier)
frontier["utility_only_nondominated"]=True
for i,row in frontier.iterrows():
    dominated=(frontier.fpr.le(row.fpr)&frontier.recall.ge(row.recall)&(frontier.fpr.lt(row.fpr)|frontier.recall.gt(row.recall))).any()
    frontier.loc[i,"utility_only_nondominated"]=not dominated
save_table(frontier,"descriptive_frontier.csv")
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for condition in CONDITIONS:
    r=frontier[frontier.condition.eq(condition)].iloc[0]
    axes[0].scatter(r.fpr,r.recall,color=colors[condition],s=65,label=labels[condition])
    axes[0].annotate(labels[condition],(r.fpr,r.recall),xytext=(5,7),textcoords="offset points",fontsize=8)
f=frontier[frontier.utility_only_nondominated].sort_values("fpr")
axes[0].plot(f.fpr,f.recall,ls="--",color="grey",alpha=.5)
axes[0].margins(x=.20,y=.18)
axes[0].set(xlabel="False Positive Rate (lower is better)",ylabel="Recall (higher is better)",title="Observed utility frontier; means only")
budget_axes(axes[1],"f1");axes[1].set(ylabel="F1",title="Formal budget versus utility")
axes[1].legend(fontsize=7);fig.tight_layout()
save_figure(fig,"09_final_tradeoff_frontier","Descriptive mean frontier among tested conditions. No universal best setting; uncertainty is shown in the companion figures.")


## 8. Verified member/nonmember distribution diagnostics

Use the three seed-42 caches already copied by Experiment 06. Every loaded file must match its
recorded byte hash and the accepted sample identities. These are representative seed-42 plots,
not repeated-seed distribution estimates. No classifier or MIA attack is fitted here.


In [9]:
distribution_frames=[]
missing_distribution_inputs=[]
distribution_sources=[]
sample=tables06["target_mia_sample_manifest.csv"]
for condition in CONDITIONS:
    rc=next(r for r in manifest06["target_run_configs"] if r["condition"]==condition and r["seed"]==42)
    path=EXP06_DIR/"intermediate"/condition/"seed_42"/"target_mia_features.csv"
    if not path.exists():
        missing_distribution_inputs.append(str(path));continue
    digest=hashlib.sha256(path.read_bytes()).hexdigest()
    require(digest==rc["output_sha256"]["mia"],f"Distribution cache checksum mismatch: {path}; restore original cache, do not retrain.")
    frame=pd.read_csv(path)
    require(frame.condition.eq(condition).all() and frame.seed.eq(42).all(),"Distribution run identity mismatch")
    pd.testing.assert_frame_equal(frame[["membership","partition_position","row_id"]].reset_index(drop=True),
        sample[["membership","partition_position","row_id"]].reset_index(drop=True),check_dtype=False)
    require(np.isfinite(frame[["confidence","loss"]]).all().all(),"Invalid distribution values")
    distribution_frames.append(frame)
    distribution_sources.append({"condition":condition,"seed":42,"path":str(path),"sha256":digest})

distributions_complete=not missing_distribution_inputs
if distributions_complete:
    fig,axes=plt.subplots(2,3,figsize=(12,7))
    cdf_rows=[]
    for col,(condition,frame) in enumerate(zip(CONDITIONS,distribution_frames)):
        for row,metric in enumerate(["confidence","loss"]):
            ax=axes[row,col]
            for membership,label,color in [(1,"Members","#087e8b"),(0,"Non-members","#c25a24")]:
                values=np.sort(frame.loc[frame.membership.eq(membership),metric].to_numpy())
                unique,counts=np.unique(values,return_counts=True)
                fraction=np.cumsum(counts)/len(values)
                ax.step(unique,fraction,where="post",label=label,color=color)
                cdf_rows.extend({"condition":condition,"seed":42,"membership":membership,"metric":metric,"value":float(x),"cdf":float(y)} for x,y in zip(unique,fraction))
            ax.set(xlabel=metric,ylabel="Empirical cumulative fraction",title=labels[condition]+"; seed 42")
            if metric=="loss":ax.set_xscale("symlog",linthresh=1e-6)
            ax.grid(alpha=.2)
    axes[0,0].legend(fontsize=8);fig.tight_layout()
    save_figure(fig,"08_member_nonmember_distributions","Hash-verified seed-42 score caches; ECDFs show confidence saturation and ties without fitting an attacker.")
    save_table(pd.DataFrame(cdf_rows),"member_nonmember_distribution_cdf.csv",PLOT_DIR)
else:
    print("Distribution figure pending. Restore the original files below; no retraining is required:")
    print("\n".join(missing_distribution_inputs))


## 9. Evidence-aligned interpretation and final export


In [10]:
findings = """# Experiment 08 interpretation boundaries

- The accepted five-seed results support final analysis, not a confirmed epsilon optimum.
- At epsilon about 4, mean Recall is 0.71338 versus 0.70934 non-private. The paired Recall
  interval crosses zero. Do not claim a confirmed detection improvement or equivalence.
- Mean FPR is 0.08755 versus 0.05610; mean average precision is 0.89068 versus 0.93750.
  The corresponding unadjusted paired intervals indicate higher FPR and lower average precision.
- MIA AUCs are close to 0.5, but several conditional across-seed intervals exclude 0.5.
  Do not repeat the Experiment 05 statement that every AUC interval contains 0.5.
- Neither primary nor secondary paired AUC comparisons establish reduced leakage.
- Epsilon about 2 has a small positive label-aware advantage difference (about 0.00345;
  unadjusted 95% interval about [0.00068, 0.00621]). Disclose it as an exploratory result.
  Advantage maximizes TPR-FPR over target thresholds; it is not the shadow-fixed operating score.
- All inference is conditional on one fixed dataset split and frozen attackers. Seed intervals
  do not include evaluation-record sampling uncertainty, shadow retraining or dataset variation.
- Unadjusted intervals span many metrics. Small positive advantage or zero-width TPR intervals
  do not establish population leakage or its absence. Raw t interval bounds are preserved,
  even when they extend outside [0,1].
- Formal DP accounting is conditional on fixed non-private preprocessing, with secure_mode=False.
  Individual-run epsilon is not a composition bound for jointly releasing every trained model.
- Epsilon 4 was identified after inspecting the sweep; this is not independent held-out
  confirmation of configuration selection. No operational acceptability threshold is established.
- Experiment 05 subgroup findings are single-seed and exploratory, especially the Rare group
  of 208 records. No new subgroup significance claims are made here.
- Keep Experiment 07, additional datasets, new privacy mechanisms and stronger attack suites
  outside this notebook. Consider external validation only after reviewing these outputs.
"""
save_text(findings,"interpretation.md")
save_table(pd.DataFrame(figure_records),"figure_captions.csv")
required_stems={f"{i:02d}" for i in range(1,10)}
observed_stems={r["figure"].split("_")[0] for r in figure_records}
core_gate=bool(len(primary_final)==6 and len(secondary_final)==6 and len(tuned)==15 and audit_report["accepted_for_final_analysis"])
figure_gate=required_stems.issubset(observed_stems)
require(core_gate,"Final analysis table gate failed")
status="COMPLETE" if distributions_complete and figure_gate else "PARTIAL_DISTRIBUTIONS_PENDING"
analysis_manifest={
    "experiment":"08_privacy_utility_frontier", "protocol_version":PROTOCOL,
    "analysis_revision":ANALYSIS_REVISION,"status":status,
    "source_experiment06_manifest_sha256":ACCEPTED_MANIFEST_SHA256,
    "source_experiment05_commit":"19e099bdc40bfd2650587c8be55659a20e9c89cc",
    "source_experiment05_snapshot_sha256":exp05_snapshot_hashes,
    "training_runs_started":0,"thresholds_tuned":0,"new_attacks_fitted":0,
    "primary_attacks":PRIMARY,"uncertainty_unit":"target-training seed; fixed records and attackers",
    "distribution_sources":distribution_sources,"missing_distribution_inputs":missing_distribution_inputs,
    "gates":{"source_evidence_gate":True,"core_tables_gate":core_gate,"required_figures_gate":figure_gate,"distribution_gate":distributions_complete},
    "software":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,"matplotlib":matplotlib.__version__},
    "output_sha256":{str(path.relative_to(OUTPUT_DIR)):hashlib.sha256(path.read_bytes()).hexdigest() for path in exports},
}
manifest_path=OUTPUT_DIR/"final_analysis_manifest.json"
manifest_path.write_text(json.dumps(analysis_manifest,indent=2),encoding="utf-8")
evidence_zip=OUTPUT_DIR/"experiment08_evidence.zip"
with zipfile.ZipFile(evidence_zip,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [*exports,manifest_path]:archive.write(path,arcname=str(path.relative_to(OUTPUT_DIR)))
with zipfile.ZipFile(evidence_zip) as archive:
    require(len(archive.namelist()) == len(set(archive.namelist())), "Duplicate export entries")
    for name,expected in analysis_manifest["output_sha256"].items():
        require(hashlib.sha256(archive.read(name)).hexdigest()==expected,"Export checksum mismatch")
print("Experiment 08 status:",status)
print("Evidence ZIP:",evidence_zip)
print("Send this ZIP for final scientific review. A partial export is not Experiment 08 completion.")
try:
    from google.colab import files
    files.download(str(evidence_zip))
except ImportError:
    pass


Experiment 08 status: COMPLETE
Evidence ZIP: /content/drive/MyDrive/ML-DP-NID/results/final_analysis/experiment08_evidence.zip
Send this ZIP for final scientific review. A partial export is not Experiment 08 completion.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>